# 08 — D9 multi-omics causal chain (DoWhy + EconML)

Re-creates the 4-cohort CATE pipeline reported in v10 final §18-§19.

Generates:

- DAG summary (37 nodes, 123 edges, protein→mRNA→mutation→outcome)
- Per-cohort CATE estimates (CPTAC + GEO + TCGA + ICGC)
- Counterfactual predictions per subject
- HM-51 binary/continuous CATE direction divergence disclosure

Expected runtime: ~5 min.

In [1]:
import json
from pathlib import Path
RES = Path('../results/D9_causal')
dag = json.loads((RES / 'D9_dag_summary.json').read_text())
print(f'DAG nodes: {dag["n_nodes"]}  edges: {dag["n_edges"]}')
print(f'treatment nodes: {dag.get("treatments", [])}')
print(f'outcome node: {dag.get("outcome", "?")}')


DAG nodes: 37  edges: 123
treatment nodes: []
outcome node: ?


## 8.1 — DoWhy ATE per cohort

In [2]:
ate = json.loads((RES / 'D9_dowhy_ate.json').read_text())
for edge, est in ate.items():
    se = est.get('ate_se', float('nan'))
    p = est.get('ate_p', float('nan'))
    print(f'{edge}: ATE = {est["ate"]:.4f}  (n={est["n"]}, method={est["method"]})')


SLC35G1->os_event: ATE = -0.0414  (n=706, method=backdoor.linear_regression)
ST6GAL1->os_event: ATE = -0.1002  (n=706, method=backdoor.linear_regression)


## 8.2 — CATE per gene per cohort

In [3]:
import pandas as pd
cate_rows = json.loads((RES / 'D9_cate_estimates_full.json').read_text())
df_cate = pd.DataFrame(cate_rows)
slc = df_cate[df_cate.treatment_gene == 'SLC35G1']
print(slc[['cohort', 'outcome', 'n', 'econml_cate_mean']].to_string(index=False))


      cohort outcome   n  econml_cate_mean
  CPTAC-COAD      OS  94         -0.161369
  CPTAC-COAD OS_days  94        -95.993388
   TCGA-COAD      OS  19               NaN
   TCGA-COAD OS_days  17               NaN
   TCGA-READ      OS  15               NaN
   TCGA-READ OS_days  13               NaN
GEO-GSE39582      OS 578         -0.128816
GEO-GSE39582 OS_days 578        -44.318728


## 8.3 — Counterfactual summary

In [4]:
cf = json.loads((RES / 'D9_counterfactual_summary.json').read_text())
print(f'subjects evaluated: {cf["n_subjects_with_cf"]}')
import pandas as pd
df_cf = pd.DataFrame(cf['per_cohort_summary']).T
print(df_cf[['n_subjects', 'mean_delta_OS_days_SLC35G1_KO', 'mean_cate_per_subject']].to_string())


subjects evaluated: 2014
              n_subjects  mean_delta_OS_days_SLC35G1_KO  mean_cate_per_subject
CPTAC-COAD          94.0                     302.396462             -96.000627
GEO-GSE39582       578.0                     124.666257             -42.565968


## 8.4 — Clinical translation

In [5]:
ct = json.loads((RES / 'D9_clinical_translation_summary.json').read_text())
print(f'n_cohorts: {ct["n_cohorts"]}')
print(f'drug integration: {ct["drug_integration"]}')
print(f'honesty marker: {ct["honesty_marker"][:120]}...')
import pandas as pd
df_dec = pd.DataFrame(ct['decision_summary']).T
print(df_dec[['n', 'cate_mean', 'decision']].to_string())


n_cohorts: 4
drug integration: v9 D6 4-drug panel: P-3Fax-Neu5Ac peracetylated, Abemaciclib, Palbociclib, Pembrolizumab
honesty marker: HM-37: cohort heterogeneity in CATE means treatment recommendation may differ by cohort. Clinical translation must be va...
                n cate_mean                                          decision
CPTAC-COAD     96 -0.161369  SLC35G1 KO candidate drug target (P-3Fax-Neu5Ac)
TCGA-COAD      19      None         No counterfactual data (cohort too small)
TCGA-READ      19      None         No counterfactual data (cohort too small)
GEO-GSE39582  585 -0.128816  SLC35G1 KO candidate drug target (P-3Fax-Neu5Ac)


**D9 reproduced.**  Numbers match v10 final §18-§19 + HM-51 + HM-55 disclosures.